# 21 · Data Modeling & Normalization

Good SQL starts with a good schema. This module covers the design theory every
engineer should know:
- entities, attributes, primary & foreign keys
- surrogate vs natural keys, composite keys
- relationship types: one-to-one, one-to-many, many-to-many
- **normalization** (1NF → 2NF → 3NF) and the anomalies it prevents
- referential integrity with `ON DELETE CASCADE`

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Keys
- **Primary key (PK):** uniquely identifies a row (e.g. `customer_id`).
- **Foreign key (FK):** a column referencing another table's PK (e.g.
  `orders.customer_id → customers.customer_id`) — this enforces *referential
  integrity*.
- **Surrogate key:** a meaningless auto-generated id (our integer ids).
- **Natural key:** a real-world unique value (e.g. an email). Surrogate keys are
  usually preferred because natural values can change.
- **Composite key:** a PK spanning multiple columns — like
  `order_items(order_id, product_id)`.

## Relationship types
- **One-to-many (1:N):** one customer → many orders. The "many" side holds the FK.
- **Many-to-many (M:N):** orders ↔ products. You resolve it with a **junction
  table** — that's exactly what `order_items` is.
- **One-to-one (1:1):** rarer; often a table split for optional/large columns.

Our whole schema in one line: `customers 1─∞ orders 1─∞ order_items ∞─1 products`.

## Why normalize? The anomalies of a "wide" table
Imagine cramming everything into one denormalized table:

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_orders_flat;
CREATE TABLE demo_orders_flat (
    order_id      INTEGER,
    customer_name TEXT,
    customer_city TEXT,     -- repeated for every order by that customer
    product_name  TEXT,
    product_price REAL,     -- repeated for every order of that product
    quantity      INTEGER
);
INSERT INTO demo_orders_flat VALUES
    (1, 'Alice', 'Seattle', 'Wireless Mouse', 25.00, 2),
    (1, 'Alice', 'Seattle', 'SQL Fundamentals', 39.99, 1),
    (2, 'Alice', 'Seattle', 'USB-C Hub', 34.50, 1);
SELECT * FROM demo_orders_flat;

This design causes three classic anomalies:
- **Update anomaly:** Alice moves city → you must update many rows or data goes
  inconsistent.
- **Insertion anomaly:** you can't record a new product until someone orders it.
- **Deletion anomaly:** delete the last order of a product and you lose the
  product's existence entirely.

Normalization removes this redundancy.

## The normal forms (informally)
- **1NF:** atomic values, no repeating groups, a key on each row. (No
  comma-separated "products" column.)
- **2NF:** 1NF **and** every non-key column depends on the *whole* composite key,
  not just part of it. (In `order_items`, `quantity` depends on both
  `order_id` **and** `product_id`; a product's `price` does **not**, so it lives
  in `products`.)
- **3NF:** 2NF **and** no *transitive* dependencies — non-key columns don't
  depend on other non-key columns. (`customer_city` depends on the customer, not
  the order, so it belongs in `customers`.)

Our course schema is already in 3NF: customer facts live in `customers`, product
facts in `products`, and `order_items` holds only what's true of a specific line
item. That's why there's no redundancy to keep in sync.

## Referential integrity: `ON DELETE CASCADE`
A foreign key can define what happens to children when a parent is deleted.
`ON DELETE CASCADE` deletes the children automatically. Foreign keys must be
enabled per connection with `PRAGMA foreign_keys = ON` (done at the top of this
one cell so the demo is self-contained):

In [ ]:
%%sql
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS demo_line_items;
DROP TABLE IF EXISTS demo_orders;

CREATE TABLE demo_orders (
    id INTEGER PRIMARY KEY,
    customer TEXT
);
CREATE TABLE demo_line_items (
    id       INTEGER PRIMARY KEY,
    order_id INTEGER REFERENCES demo_orders(id) ON DELETE CASCADE,
    item     TEXT
);

INSERT INTO demo_orders VALUES (1, 'Alice'), (2, 'Bob');
INSERT INTO demo_line_items (order_id, item) VALUES (1, 'Mouse'), (1, 'Book'), (2, 'Pan');

-- delete order 1: its line items vanish automatically
DELETE FROM demo_orders WHERE id = 1;

SELECT * FROM demo_line_items;

Only Bob's line item remains — the cascade removed order 1's children.

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_line_items;
DROP TABLE IF EXISTS demo_orders;
DROP TABLE IF EXISTS demo_orders_flat;
SELECT 'cleaned up' AS status;

## Practice

**✏️ Exercise 1.** In our schema, which table resolves the many-to-many relationship between orders and products, and what is its composite primary key? (Write a query listing that table's columns to confirm.)

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
PRAGMA table_info(order_items);

### ✅ Recap
Model entities with PKs, connect them with FKs, resolve M:N with junction tables,
and normalize to 3NF to eliminate update/insert/delete anomalies. Use
`ON DELETE CASCADE` (with `PRAGMA foreign_keys = ON`) to keep children consistent
with parents.

**Next:** `22_indexing_and_performance.ipynb`.